# Fase 2: Preprocesamiento, Calidad y Limpieza de Datos
**Proyecto:** Análisis del desempeño SIMCE 2025 (4° Básico - Matemática)
**Objetivo F2:** Cargar la base bruta del SIMCE 2025, aplicar filtros de efectividad operacional, enriquecer el dataset con taxonomías geográficas y administrativas (región, provincia, macrozona, GSE, ruralidad), estandarizar tipos de datos y exportar el dataset limpio y procesado para la Fase 3.

---
### Contenido del Pipeline F2:
1. Configuración del entorno y rutas dinámicas.
2. Definición de mapeos y diccionarios cualitativos.
3. Extracción de datos crudos (Raw Data).
4. Estandarización de tipos numéricos.
5. Filtro de efectividad operacional (exclusión de registros inválidos).
6. Enriquecimiento geográfico y taxonómico.
7. Limpieza y normalización de campos de texto.
8. Renombrado y estandarización de columnas.
9. Validación de calidad del dataset final (Assertions).
10. Exportación del dataset procesado.

In [1]:
import sys
from pathlib import Path
import pandas as pd

# 1. DETECCIÓN DINÁMICA DE LA RAÍZ DEL REPOSITORIO
def obtener_raiz_repositorio(directorio_actual: Path = Path.cwd()) -> Path:
    directorio = directorio_actual.resolve()
    for parent in [directorio] + list(directorio.parents):
        if (parent / ".git").exists() or (parent / "requirements.txt").exists():
            return parent
    return directorio_actual

RAIZ = obtener_raiz_repositorio()
RAW_FILE = RAIZ / "data" / "raw" / "simce4b2025_rbd_final.csv"
OUTPUT_DIR = RAIZ / "data" / "processed"
OUTPUT_FILE = OUTPUT_DIR / "simce4b2025_matematica_efectiva.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Directorio Raíz: {RAIZ}")
print(f"Ruta Archivo Raw: {RAW_FILE}")
print(f"Ruta Destino Processed: {OUTPUT_FILE}")
assert RAW_FILE.exists(), f"Error: No se encuentra el archivo de entrada en {RAW_FILE}"


Directorio Raíz: C:\Users\jorge\f1_s01_evaluacion_entregable_grupo7
Ruta Archivo Raw: C:\Users\jorge\f1_s01_evaluacion_entregable_grupo7\data\raw\simce4b2025_rbd_final.csv
Ruta Destino Processed: C:\Users\jorge\f1_s01_evaluacion_entregable_grupo7\data\processed\simce4b2025_matematica_efectiva.csv


## 2. Diccionarios de Categorías y Mapeos Geográficos
Definición de las estructuras auxiliares para la codificación y categorización de dependencias administrativas, grupos socioeconómicos (GSE), zonas de ruralidad y ordenamiento macrozonal de Chile (Norte a Sur).

In [2]:
# Mapeo de Administración Escolar
DEPENDENCIA_1 = {
    1: "Municipal Corporación",
    2: "Municipal DAEM",
    3: "Particular subvencionado",
    4: "Particular pagado",
    5: "Corporación de administración delegada",
    6: "Servicio Local de Educación",
}

DEPENDENCIA_2 = {
    1: "Municipal",
    2: "Particular subvencionado",
    3: "Particular pagado",
    4: "Servicio Local de Educación",
}

# Mapeos Socioeconómicos y Entorno
GSE = {
    1: "Bajo",
    2: "Medio bajo",
    3: "Medio",
    4: "Medio alto",
    5: "Alto",
}

RURALIDAD = {
    1: "Urbano",
    2: "Rural",
}

# Geografía y Macrozonas de Chile (Orden de Norte a Sur)
GEOGRAFIA_REGION = {
    15: ("Norte", "Norte Grande", 1),   # Arica y Parinacota
    1:  ("Norte", "Norte Grande", 2),   # Tarapacá
    2:  ("Norte", "Norte Grande", 3),   # Antofagasta
    3:  ("Norte", "Norte Chico",  4),   # Atacama
    4:  ("Norte", "Norte Chico",  5),   # Coquimbo
    5:  ("Centro", "Zona Central", 6),  # Valparaíso
    13: ("Centro", "Zona Central", 7),  # Metropolitana
    6:  ("Centro", "Zona Central", 8),  # O'Higgins
    7:  ("Centro", "Zona Central", 9),  # Maule
    16: ("Sur",    "Zona Sur",    10),  # Ñuble
    8:  ("Sur",    "Zona Sur",    11),  # Biobío
    9:  ("Sur",    "Zona Sur",    12),  # Araucanía
    14: ("Sur",    "Zona Sur",    13),  # Los Ríos
    10: ("Sur",    "Zona Sur",    14),  # Los Lagos
    11: ("Sur",    "Zona Austral", 15), # Aysén
    12: ("Sur",    "Zona Austral", 16), # Magallanes
}

print("Diccionarios y mapeos cargados correctamente en memoria.")

Diccionarios y mapeos cargados correctamente en memoria.


## 3. Extracción de Datos Crudos (Raw Load)
Carga selectiva de columnas necesarias desde el archivo plano original (`.csv` codificado en `CP1252` con separador `;`). Esto optimiza el uso de memoria RAM evitando la carga de variables prescindibles.

In [3]:
COLUMNAS_INPUT = [
    "rbd", "dvrbd", "nom_rbd",
    "cod_reg_rbd", "nom_reg_rbd",
    "cod_pro_rbd", "nom_pro_rbd",
    "cod_com_rbd", "nom_com_rbd",
    "cod_deprov_rbd", "nom_deprov_rbd",
    "cod_depe1", "cod_depe2", "cod_grupo", "cod_rural_rbd",
    "nalu_mate4b_rbd", "prom_mate4b_rbd",
    "palu_eda_ins_mate4b_rbd", "palu_eda_ele_mate4b_rbd", "palu_eda_ade_mate4b_rbd",
    "marca_mate4b_rbd", "noaplica", "codigo_bbdd", "fecha_bbdd", "grado", "agno"
]

df_raw = pd.read_csv(
    RAW_FILE,
    sep=";",
    encoding="cp1252",
    usecols=COLUMNAS_INPUT,
    low_memory=False
)

print(f"Dimensiones preliminares: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas.")
# Conservamos una referencia del bruto completo para comparar sus nulos con la salida.
df_bruto_completo = pd.read_csv(RAW_FILE, sep=";", encoding="cp1252", low_memory=False)


Dimensiones preliminares: 7,143 filas x 26 columnas.


## 4. Coerción y Estandarización Numérica
Conversión explícita de campos a tipos de datos numéricos. Se conservan los nulos originales y se detiene la ejecución si aparecen valores no convertibles, para evitar perder información silenciosamente.

También convertimos `fecha_bbdd` desde AAAAMMDD a una fecha: por ejemplo, 20260622 pasa a 2026-06-22. Es una fecha de la base y es constante en este archivo; no representa distintas fechas de evaluación ni permite estudiar tendencias. El criterio del 60% de nulos se evalúa sobre el bruto y esta conversión no cambia ese resultado.

In [4]:
columnas_numericas = [
    "rbd", "dvrbd", "cod_reg_rbd", "cod_pro_rbd", "cod_com_rbd",
    "cod_deprov_rbd", "cod_depe1", "cod_depe2", "cod_grupo", "cod_rural_rbd",
    "nalu_mate4b_rbd", "prom_mate4b_rbd", "palu_eda_ins_mate4b_rbd",
    "palu_eda_ele_mate4b_rbd", "palu_eda_ade_mate4b_rbd", "marca_mate4b_rbd",
    "noaplica", "agno"
]

for col in columnas_numericas:
    convertida = pd.to_numeric(df_raw[col], errors="coerce")
    invalidos = df_raw[col].notna() & convertida.isna()
    assert not invalidos.any(), f"Valores no numéricos en {col}: {int(invalidos.sum())}"
    df_raw[col] = convertida

print("Transformación a tipos numéricos completada.")

# La fecha tiene ocho dígitos, pero no es una cantidad: la convertimos por separado.
# El formato explícito evita interpretarla como un número de días o segundos.
fecha_texto = df_raw["fecha_bbdd"].astype("string").str.strip()
assert fecha_texto.str.fullmatch(r"[0-9]{8}").fillna(False).all(), "Revisar fecha_bbdd: se espera AAAAMMDD sin vacíos."
df_raw["fecha_bbdd"] = pd.to_datetime(fecha_texto, format="%Y%m%d", errors="raise")
assert df_raw["fecha_bbdd"].notna().all(), "Hay fechas de base ausentes."
print("Fecha de la base:", df_raw["fecha_bbdd"].dt.strftime("%Y-%m-%d").unique())


Transformación a tipos numéricos completada.
Fecha de la base: <StringArray>
['2026-06-22']
Length: 1, dtype: str


## 5. Filtro de Efectividad Operacional
Se aplica el criterio de selección utilizado por el equipo para aislar los establecimientos con mediciones válidas y efectivas:
* Número de alumnos evaluados mayor a cero (`nalu_mate4b_rbd > 0`).
* Ausencia de códigos de observación/exclusión en la marca (`marca_mate4b_rbd` no pertenece a `[1, 2, 3, 4]`).

El significado de las marcas debe contrastarse con el diccionario oficial. Una marca vacía se admite según el criterio actual; no se interpreta automáticamente como un dato erróneo.

Los porcentajes de niveles de aprendizaje se conservan como información complementaria. Sus ausencias no eliminan establecimientos que tienen puntaje válido, ya que la problemática se centra en comparar puntajes por territorio. No se rellenan con cero ni se imputan resultados no disponibles.

Antes de filtrar mostramos cuántos registros tiene cada marca y cuántos conservan puntaje. En esta base, la marca 1 aparece en 450 registros sin puntaje y la marca 2 en 55 con puntaje. Por eso no asumimos que todas las marcas signifiquen lo mismo. Mantenemos la regla del grupo hasta confirmar las glosas de 2025; los 55 casos con marca 2 son el punto concreto que queda por revisar.

El portal oficial consultado es https://informacionestadistica.agenciaeducacion.cl/#/bases. No se pudo verificar allí la tabla de códigos exacta de esta versión, por lo que no atribuimos significados oficiales a partir de las frecuencias.

In [5]:
# Revisamos el efecto de los códigos antes de excluir filas.
# Estas tablas describen el archivo; no reemplazan las glosas oficiales.
for columna in ["marca_mate4b_rbd", "noaplica"]:
    revision = df_raw.groupby(columna, dropna=False).agg(
        establecimientos=("rbd", "size"),
        con_puntaje=("prom_mate4b_rbd", "count")
    )
    revision["sin_puntaje"] = revision["establecimientos"] - revision["con_puntaje"]
    print(f"Revisión de {columna}:")
    display(revision)

# Dejamos la lista en una variable para cambiarla en un solo lugar si las glosas lo indican.
MARCAS_EXCLUIDAS = [1, 2, 3, 4]
filas_antes = len(df_raw)

df_efectivo = df_raw.loc[
    df_raw["nalu_mate4b_rbd"].gt(0) &
    ~df_raw["marca_mate4b_rbd"].isin(MARCAS_EXCLUIDAS)
].copy()

filas_despues = len(df_efectivo)
print(f"Registros originales: {filas_antes:,}")
print(f"Registros filtrados (Efectivos): {filas_despues:,}")
print(f"Registros excluidos: {filas_antes - filas_despues:,}")

assert filas_despues > 0, "Error: El proceso de filtrado descartó la totalidad de los datos."

# Estas causas pueden coincidir en un mismo establecimiento.
print("Sin alumnos evaluados:", int((~df_raw["nalu_mate4b_rbd"].gt(0)).sum()))
print("Con marca excluida:", int(df_raw["marca_mate4b_rbd"].isin(MARCAS_EXCLUIDAS).sum()))

# Hacemos visible la parte de la decisión que requiere respaldo documental.
con_puntaje_excluidos = df_raw["marca_mate4b_rbd"].isin(MARCAS_EXCLUIDAS) & df_raw["prom_mate4b_rbd"].notna()
print("Con puntaje disponible pero excluidos por marca:", int(con_puntaje_excluidos.sum()))
print("Códigos noaplica presentes después del filtro:")
display(df_efectivo["noaplica"].value_counts(dropna=False))


Revisión de marca_mate4b_rbd:


,establecimientos,con_puntaje,sin_puntaje
marca_mate4b_rbd,,,
1.0,450,0,450
2.0,55,55,0
NaN,6638,6524,114


Revisión de noaplica:


,establecimientos,con_puntaje,sin_puntaje
noaplica,,,
0,7029,6579,450
1,82,0,82
2,29,0,29
3,3,0,3


Registros originales: 7,143
Registros filtrados (Efectivos): 6,524
Registros excluidos: 619
Sin alumnos evaluados: 125
Con marca excluida: 505
Con puntaje disponible pero excluidos por marca: 55
Códigos noaplica presentes después del filtro:


noaplica
0    6524
Name: count, dtype: int64

## 6. Enriquecimiento Taxonómico y Geográfico
Aplicación de las jerarquías territoriales y cualitativas mediante los mapeos previamente cargados. Se construyen cadenas compuestas de ubicación georeferenciada.
Los nombres se limpian antes de concatenar las ubicaciones, para que estas también queden normalizadas.

In [6]:
# 1. Limpieza de textos
columnas_texto = [
    "nom_rbd", "nom_reg_rbd", "nom_pro_rbd", 
    "nom_com_rbd", "nom_deprov_rbd", "codigo_bbdd", "grado"
]

for col in columnas_texto:
    df_efectivo[col] = df_efectivo[col].astype("string").str.strip().replace("", pd.NA)

# Comprobar que los códigos disponibles tienen una etiqueta definida.
for columna, catalogo in [("cod_depe1", DEPENDENCIA_1), ("cod_depe2", DEPENDENCIA_2),
                          ("cod_grupo", GSE), ("cod_rural_rbd", RURALIDAD),
                          ("cod_reg_rbd", GEOGRAFIA_REGION)]:
    desconocidos = set(df_efectivo[columna].dropna()) - set(catalogo)
    assert not desconocidos, f"Códigos sin correspondencia en {columna}: {desconocidos}"

# Mapeo de variables categóricas
df_efectivo["dependencia_6_cat"] = df_efectivo["cod_depe1"].map(DEPENDENCIA_1)
df_efectivo["dependencia_4_cat"] = df_efectivo["cod_depe2"].map(DEPENDENCIA_2)
df_efectivo["grupo_socioeconomico"] = df_efectivo["cod_grupo"].map(GSE)
df_efectivo["ruralidad"] = df_efectivo["cod_rural_rbd"].map(RURALIDAD)

# Enriquecimiento geográfico
geo = df_efectivo["cod_reg_rbd"].map(GEOGRAFIA_REGION)

df_efectivo["Zona"] = geo.str[0]
df_efectivo["Macrozona"] = geo.str[1]
df_efectivo["Orden"] = geo.str[2].astype("Int64")
df_efectivo["pais"] = "Chile"

# Concatenación de ubicaciones jerárquicas
df_efectivo["ubicacion_region"] = df_efectivo["nom_reg_rbd"].astype("string") + ", Chile"
df_efectivo["ubicacion_provincia"] = (
    df_efectivo["nom_pro_rbd"].astype("string") + ", " + 
    df_efectivo["nom_reg_rbd"].astype("string") + ", Chile"
)
df_efectivo["ubicacion_comuna"] = (
    df_efectivo["nom_com_rbd"].astype("string") + ", " + 
    df_efectivo["nom_reg_rbd"].astype("string") + ", Chile"
)

print("Variables taxonómicas y geográficas generadas con éxito.")

Variables taxonómicas y geográficas generadas con éxito.


## 7. Renombrado de Columnas
Traducción de nombres de columnas técnicos hacia nombres autoexplicativos. Los textos ya se normalizaron antes de construir las ubicaciones.

In [7]:
# 2. Renombrar columnas a estándar final
df_efectivo = df_efectivo.rename(
    columns={
        "dvrbd": "dv_rbd",
        "nom_rbd": "nombre_establecimiento",
        "nom_reg_rbd": "region",
        "nom_pro_rbd": "provincia",
        "nom_com_rbd": "comuna",
        "nom_deprov_rbd": "deprov",
        "nalu_mate4b_rbd": "n_alumnos",
        "prom_mate4b_rbd": "puntaje_promedio",
        "palu_eda_ins_mate4b_rbd": "pct_insuficiente",
        "palu_eda_ele_mate4b_rbd": "pct_elemental",
        "palu_eda_ade_mate4b_rbd": "pct_adecuado",
        "noaplica": "no_aplica",
        "agno": "anio",
    }
)

# 3. Variables constantes complementarias
df_efectivo["asignatura"] = "Matematica"
df_efectivo["efectividad"] = "Efectiva"

print("Estandarización de columnas completada.")

Estandarización de columnas completada.


## 8. Selección de Columnas y Validaciones de Calidad (Assertions)
Se genera la vista definitiva descartando columnas intermedias o de control técnico. Posteriormente se aplican pruebas unitarias de calidad para garantizar la integridad del dataset procesado.
Las marcas se utilizaron para la selección, pero no forman parte de las columnas finales. Conservamos los porcentajes incompletos como información complementaria y los puntajes en sus unidades originales.

In [8]:
COLUMNAS_FINALES = [
    "rbd", "dv_rbd", "nombre_establecimiento", "asignatura",
    "cod_reg_rbd", "region", "cod_pro_rbd", "provincia", "cod_com_rbd", "comuna", "deprov",
    "pais", "ubicacion_region", "ubicacion_provincia", "ubicacion_comuna",
    "Zona", "Macrozona", "Orden",
    "dependencia_6_cat", "dependencia_4_cat", "grupo_socioeconomico", "ruralidad",
    "n_alumnos", "puntaje_promedio",
    "pct_insuficiente", "pct_elemental", "pct_adecuado",
    "efectividad", "no_aplica", "codigo_bbdd", "fecha_bbdd", "grado", "anio"
]

df_final = df_efectivo[COLUMNAS_FINALES].copy()

# VALIDACIONES DE INTEGRIDAD (F2)
assert not df_final.empty, "ERROR: El dataset procesado quedó vacío."
assert df_final["asignatura"].eq("Matematica").all(), "ERROR: Hay registros con asignaturas distintas a Matemática."
assert df_final["efectividad"].eq("Efectiva").all(), "ERROR: Hay registros etiquetados como No Efectivos."
assert df_final["n_alumnos"].gt(0).all(), "ERROR: Hay registros con n_alumnos <= 0."
assert df_final.columns.duplicated().sum() == 0, "ERROR: Se detectaron columnas duplicadas en la salida."
assert df_final["rbd"].notna().all(), "ERROR: Existen registros con identificador RBD nulo."

assert df_final["rbd"].is_unique, "ERROR: Hay establecimientos repetidos."
assert df_final[["puntaje_promedio", "region", "provincia", "comuna"]].notna().all().all(), "ERROR: Faltan puntajes o datos territoriales para la comparacion."

# Códigos necesarios para agrupar por territorio sin depender del nombre.
assert df_final[["cod_reg_rbd", "cod_pro_rbd", "cod_com_rbd"]].notna().all().all(), "Faltan códigos territoriales."

# Los porcentajes pueden faltar; cuando existen deben ser coherentes.
columnas_pct = ["pct_insuficiente", "pct_elemental", "pct_adecuado"]
porcentajes = df_final[columnas_pct]
for col in columnas_pct:
    assert porcentajes[col].dropna().between(0, 100).all(), f"Porcentajes fuera de rango en {col}"
completos = porcentajes.dropna()
assert completos.sum(axis=1).sub(100).abs().le(0.15).all(), "Los porcentajes no suman 100 dentro del redondeo esperado."
print("Establecimientos sin los tres porcentajes:", int(porcentajes.isna().all(axis=1).sum()))
print("Establecimientos con porcentajes parcialmente disponibles:", int(porcentajes.isna().sum(axis=1).isin([1, 2]).sum()))

print("--- TODAS LAS VALIDACIONES DE CALIDAD FUE PASADAS CON ÉXITO ---")
print(f"Estructura Final: {df_final.shape[0]:,} filas x {df_final.shape[1]} columnas.")


Establecimientos sin los tres porcentajes: 1446
Establecimientos con porcentajes parcialmente disponibles: 0
--- TODAS LAS VALIDACIONES DE CALIDAD FUE PASADAS CON ÉXITO ---
Estructura Final: 6,524 filas x 33 columnas.


## 9. Exportación del Dataset Procesado
Almacenamiento del archivo final en formato `.csv` codificado en `UTF-8 con BOM` (`utf-8-sig`) para garantizar compatibilidad con sistemas externos y plataformas analíticas (Excel, Power BI, Python/Pandas).

In [9]:
df_final.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"  # Guardamos la fecha de forma legible en el CSV.
)

print("Pipeline F2 finalizado exitosamente.")
print(f"Archivo persistido en: {OUTPUT_FILE}")
# Comprobar que la exportación conserva el contenido de la tabla.
releido = pd.read_csv(OUTPUT_FILE, encoding="utf-8-sig")
# CSV guarda texto: recuperamos el tipo fecha para comprobar la ida y vuelta.
releido["fecha_bbdd"] = pd.to_datetime(releido["fecha_bbdd"], format="%Y-%m-%d", errors="raise")
pd.testing.assert_frame_equal(
    releido, df_final.reset_index(drop=True), check_dtype=False
)
print("Archivo releído y verificado correctamente.")

# Comparamos las ausencias antes y después, sin confundir ambas versiones del dataset.
resumen_nulos = pd.DataFrame([
    {"dataset": "Bruto completo", "filas": len(df_bruto_completo), "columnas": len(df_bruto_completo.columns),
     "max_nulos_pct": df_bruto_completo.isna().mean().max() * 100},
    {"dataset": "Procesado", "filas": len(df_final), "columnas": len(df_final.columns),
     "max_nulos_pct": df_final.isna().mean().max() * 100},
])
display(resumen_nulos)


Pipeline F2 finalizado exitosamente.
Archivo persistido en: C:\Users\jorge\f1_s01_evaluacion_entregable_grupo7\data\processed\simce4b2025_matematica_efectiva.csv
Archivo releído y verificado correctamente.


,dataset,filas,columnas,max_nulos_pct
0,Bruto completo,7143,42,92.930141
1,Procesado,6524,33,22.164316


### Cierre de esta revisión
La salida conserva 6.524 establecimientos y 33 columnas. Los porcentajes faltan en 1.446 filas, pero sus puntajes y ubicación están disponibles. El máximo de nulos pasa de 92,93% en el bruto completo a aproximadamente 22,16% en la salida.

No rellenamos las marcas para reducir ese porcentaje ni presentamos la salida como si fuera el archivo original. Nos falta confirmar las glosas de las marcas, especialmente los 55 casos con marca 2, y acordar con el docente cómo se considera el umbral del bruto. El resto de esta revisión corrige cifras y deja la selección más clara, sin cambiar los establecimientos elegidos.